# Gerar pela API ou conferir uma proposta

POC DEMO — contrato PROVISÓRIO / PENDENTE DE HOMOLOGAÇÃO. Execute em kernel novo. Credenciais somente por ambiente.


Configuração tipada. Pré-requisitos: Docker saudável, seed DEMO, token por ambiente. Entradas: parâmetros abaixo; saídas: tabelas, IDs e relatório JSON/Markdown. Escritas exigem confirmação EXECUTE_DEMO e chave estável; repetir chave igual recupera o mesmo job. Espere READY; erros de dependência exigem verificar docker compose logs.

In [1]:
import os, json
import pandas as pd
from IPython.display import display, Markdown
from src.workbench.journey_config import Settings
cfg=Settings.load()
from src.workbench.notebook_config import Parameters
from src.workbench.operations_sdk import call, report, await_job
params=Parameters.load()
display(params.model_dump(mode="json"))
from src.workbench.manual_validation import display_locations, audit, export_session
display({"state":"READY", "write_enabled":params.execute_write, "tracking":"remote Docker service", "browser_ui":cfg.ui})
evidence={}
NOTEBOOK_ID="continuation-auto-check"


{'mode': 'READ_EXISTING',
 'proposal_id': None,
 'snapshot_id': None,
 'model_name': 'planilha-turmas-demo',
 'model_version': '',
 'registry_prefix': '',
 'registry_page_token': '',
 'campaign_id': None,
 'approval_id': None,
 'execute_write': False,
 'confirmation': '',
 'request_key': '',
 'requester': 'scientist-demo',
 'algorithm': 'ridge',
 'alpha': 1.0,
 'capacity': 40,
 'model_count': 3,
 'campaign_display_name': 'POC DEMO • Validação manual',
 'parallel_jobs': 2,
 'campaign_action': 'observe',
 'governance_action': 'read',
 'review_action': 'APPROVE',
 'reason': '',
 'lineage_kind': 'proposal',
 'external_id': '',
 'reset_mode': 'preview',
 'reinitialize': 'demo-ready',
 'reset_plan_id': None}

{'state': 'READY',
 'write_enabled': False,
 'tracking': 'remote Docker service',
 'browser_ui': 'http://127.0.0.1:5000'}

1. Preflight e seleção. READ_EXISTING não escreve; GENERATE_AND_VALIDATE envia POST ao worker. Escolha explicitamente snapshot e modelo/versão; confirme a escrita e defina request_key.

In [2]:
from src.workbench.contract_review import preflight,candidates,fetch_payload,reconcile
display(pd.DataFrame(preflight(cfg)))
available_proposals=pd.DataFrame(candidates(cfg)).rename(columns={"execution_id":"proposal_id"})
display(available_proposals)
if available_proposals.empty:
    display(Markdown("Não há propostas neste ciclo. Execute a jornada 01 ou configure GENERATE_AND_VALIDATE abaixo com um modelo registrado e snapshot explícitos."))


,dependency,status
0,API/OpenAPI/auth negative/MLflow SDK,PASS


,proposal_id,generated_at,model_name,model_version,status,row_count
0,69608bc2-04bb-40e2-ac79-e6a0ce3c3570,2026-11-05 03:02:52.406025+00:00,planilha-turmas-demo,18,DRAFT,90
1,1ff2ee34-c872-47ae-a166-ae12d8f24ac5,2026-11-05 02:47:40.348895+00:00,planilha-turmas-demo,17,DRAFT,90
2,e76704c2-c5cd-4481-ba7a-9367a6ac7ba7,2026-11-04 21:48:22.229189+00:00,hi.demo.integration.be61af4a,1,DRAFT,90
3,ca4539df-4a11-4647-9d77-2cc36a484968,2026-11-04 21:48:15.351437+00:00,hi.pt.demanda.DEMO-C1.DEMO-CR1.P1,4,DRAFT,9
4,f20b5694-508e-4ea8-bcc1-3a8b55a8d528,2026-11-04 21:31:50.034018+00:00,hi.demo.integration.b8cf3663,1,DRAFT,90
5,d6035f5d-fe5c-46d5-9913-c26d661c8d3b,2026-11-04 21:31:43.053407+00:00,hi.pt.demanda.DEMO-C1.DEMO-CR1.P1,2,DRAFT,9
6,7f3923d2-8b69-4d16-9ccd-169bd658080a,2026-11-04 21:20:32.082174+00:00,planilha-turmas-demo,16,DRAFT,90
7,3c047abe-c2f3-44df-ad4d-157a8d61feb8,2026-11-04 20:12:09.721612+00:00,hi.demo.integration.5a00d837,1,DRAFT,90
8,8ba35d3e-70ee-4719-943a-4b607d2549ee,2026-11-04 19:29:06.312865+00:00,planilha-turmas-demo,15,DRAFT,90
9,896ad0e3-1a39-4c77-890b-0c622aad76f3,2026-11-04 19:24:44.231085+00:00,hi.demo.integration.d0b9bf81,1,DRAFT,90


1.1 Continuidade automática. Deixe selection vazio para conferir a proposta da jornada 01 concluída. O contexto é validado na API e exibido abaixo; não se pesquisa a proposta mais recente no banco. Para escolher outra proposta, preencha proposal_id com o UUID da tabela. READ_EXISTING consulta e reconcilia, sem nova inferência. Para gerar outra proposta, preencha o segundo exemplo com os IDs do treino 04/05. Configuração salva permite Restart Kernel and Run All.

In [3]:
# Vazio: continua a jornada 01; parâmetros explícitos do ambiente têm prioridade.
selection = {
    # "mode": "READ_EXISTING",
    # "proposal_id": "COLE_O_UUID_DA_TABELA",
}
# Alternativa: substitua selection acima por este dicionário preenchido.
# selection = {
#     "mode": "GENERATE_AND_VALIDATE",
#     "snapshot_id": "COLE_O_SNAPSHOT_ID_DO_TREINO",
#     "model_name": "COLE_O_NOME_REGISTRADO",
#     "model_version": "1",  # versão escolhida explicitamente
#     "execute_write": True,
#     "confirmation": "EXECUTE_DEMO",
#     "request_key": "meu-teste-inferencia-001",
# }
# Mesma chave + mesmo conteúdo: recupera o job. Nova chave: nova proposta.
params=Parameters.model_validate({**Parameters.load().model_dump(), **selection})
proposal_id=str(params.proposal_id) if params.proposal_id else ""
display(params.model_dump(mode="json"))
if params.mode=="READ_EXISTING" and not proposal_id:
    from src.workbench.continuation import resolve_proposal
    proposal_id,continuation=resolve_proposal(cfg)
    display(continuation)
    evidence["continuation"]=continuation


{'mode': 'READ_EXISTING',
 'proposal_id': None,
 'snapshot_id': None,
 'model_name': 'planilha-turmas-demo',
 'model_version': '',
 'registry_prefix': '',
 'registry_page_token': '',
 'campaign_id': None,
 'approval_id': None,
 'execute_write': False,
 'confirmation': '',
 'request_key': '',
 'requester': 'scientist-demo',
 'algorithm': 'ridge',
 'alpha': 1.0,
 'capacity': 40,
 'model_count': 3,
 'campaign_display_name': 'POC DEMO • Validação manual',
 'parallel_jobs': 2,
 'campaign_action': 'observe',
 'governance_action': 'read',
 'review_action': 'APPROVE',
 'reason': '',
 'lineage_kind': 'proposal',
 'external_id': '',
 'reset_mode': 'preview',
 'reinitialize': 'demo-ready',
 'reset_plan_id': None}

{'selection_source': '/reports/journey-latest.json',
 'proposal_id': '1ff2ee34-c872-47ae-a166-ae12d8f24ac5',
 'run_id': '6ac7e51b7b52456cb9b2230152691bed',
 'model_name': 'planilha-turmas-demo',
 'model_version': '17',
 'snapshot_id': 'fb931e1e-43d0-5ff8-938c-0d2ba4a68794'}

2. Submeter geração somente no modo opt-in. Nenhuma persistência é feita pelo notebook. Esperado: job, correlation_id e status_url.

In [4]:
if params.mode=="GENERATE_AND_VALIDATE":
    params.authorize()
    if not params.snapshot_id or not params.model_version:
        raise ValueError("Selecione snapshot_id e model_version explicitamente")
    from src.workbench.operations_sdk import generate
    payload=dict(cycle_code=cfg.cycle,snapshot_id=str(params.snapshot_id),
        model_selectors=[dict(model_name=params.model_name,model_version=params.model_version)],
        capacity=params.capacity,requested_by=params.requester,confirm_demo=True)
    display(payload)
    submitted=generate(cfg,payload,params.request_key)
    display(submitted)
    finished=await_job(cfg,submitted["status_url"])
    if finished["status"]!="COMPLETED":
        raise RuntimeError("Geração não completou; consulte o job persistido: "+str(finished))
    proposal_id=finished["proposal_id"]
    evidence.update(request=payload,job=finished)
if not proposal_id:
    raise ValueError("Selecione proposal_id da tabela ou configure modo GENERATE_AND_VALIDATE")


3. Obter payload integral paginado e validar schema/hash. Este é o JSON consumido pela aplicação.

In [5]:
pages,offers=fetch_payload(cfg,proposal_id,page_size=50)
for page in pages: display(page)
display(pd.json_normalize(offers))
display(pd.DataFrame(pages[0]["validation"]))
assert not any(r["status"]=="FAIL" for r in pages[0]["validation"])
evidence.update(proposal_id=proposal_id,http_pages=pages)


{'schema_version': 'poc-planilha-turmas-v1',
 'contract_status': 'PROVISÓRIO / PENDENTE DE HOMOLOGAÇÃO',
 'proposal_id': '1ff2ee34-c872-47ae-a166-ae12d8f24ac5',
 'execution_id': '1ff2ee34-c872-47ae-a166-ae12d8f24ac5',
 'proposal_revision': 1,
 'academic_cycle': {'academic_cycle_id': '0fa6938d-2743-56ae-a057-dffd2274d90a',
  'cycle_code': '2026-1'},
 'status': 'DRAFT',
 'source_type': 'DEMO',
 'generated_at_utc': '2026-11-05T02:47:40.348895Z',
 'lineage': {'snapshot_id': 'fb931e1e-43d0-5ff8-938c-0d2ba4a68794',
  'snapshot_sha256': '7b4120a14004aeb7cc006c0e210f004a0a6dffd9510bf304f27641b76dbd377e',
  'dataset_sha256': '5fc39eeeead59aa48813182e6a93d2308a4a9243655501f17d09a981c6c66a56',
  'training_campaign_id': None,
  'ruleset_version': 'POC-whole-cohort-ceil-capacity-v1',
  'model_usages': [{'usage_id': 'c4420ce1-ac23-4538-874d-aeb7c2f67c2b',
    'proposal_id': '1ff2ee34-c872-47ae-a166-ae12d8f24ac5',
    'inference_id': 'ada1ffcb-9146-4be8-9437-deb9f455aa20',
    'model_key': 'planilha-

{'schema_version': 'poc-planilha-turmas-v1',
 'contract_status': 'PROVISÓRIO / PENDENTE DE HOMOLOGAÇÃO',
 'proposal_id': '1ff2ee34-c872-47ae-a166-ae12d8f24ac5',
 'execution_id': '1ff2ee34-c872-47ae-a166-ae12d8f24ac5',
 'proposal_revision': 1,
 'academic_cycle': {'academic_cycle_id': '0fa6938d-2743-56ae-a057-dffd2274d90a',
  'cycle_code': '2026-1'},
 'status': 'DRAFT',
 'source_type': 'DEMO',
 'generated_at_utc': '2026-11-05T02:47:40.348895Z',
 'lineage': {'snapshot_id': 'fb931e1e-43d0-5ff8-938c-0d2ba4a68794',
  'snapshot_sha256': '7b4120a14004aeb7cc006c0e210f004a0a6dffd9510bf304f27641b76dbd377e',
  'dataset_sha256': '5fc39eeeead59aa48813182e6a93d2308a4a9243655501f17d09a981c6c66a56',
  'training_campaign_id': None,
  'ruleset_version': 'POC-whole-cohort-ceil-capacity-v1',
  'model_usages': [{'usage_id': 'c4420ce1-ac23-4538-874d-aeb7c2f67c2b',
    'proposal_id': '1ff2ee34-c872-47ae-a166-ae12d8f24ac5',
    'inference_id': 'ada1ffcb-9146-4be8-9437-deb9f455aa20',
    'model_key': 'planilha-

,offer_id,campus_id,course_id,period_id,discipline_id,cohort,shift,offer_type,status,forecast_raw,...,students.regular,students.dependency,students.elective,students.total,students.origins,modulation.rule_version,modulation.capacity,modulation.proposed_groups,modulation.groups,modulation.provisional
0,c268226e-6532-5927-9ed5-f2f6ae27177b,7db3550a-82e7-573e-a7c7-1876dea74c09,0169f756-26d1-5ad6-acd4-4e6b0e319939,39c82c86-d06e-5f26-9712-a92b4018084e,172d26b5-b285-5ec4-8971-f3f1cac5befb,None,None,None,SUGERIDA,102.089955,...,None,None,None,103,None,POC-whole-cohort-ceil-capacity-v1,40,3,None,True
1,c451556d-14d9-5d86-be9d-568423b8d7b7,7db3550a-82e7-573e-a7c7-1876dea74c09,0169f756-26d1-5ad6-acd4-4e6b0e319939,39c82c86-d06e-5f26-9712-a92b4018084e,3c7c40f4-7b48-5e65-ad02-fdf7381ae7a3,None,None,None,SUGERIDA,102.089955,...,None,None,None,103,None,POC-whole-cohort-ceil-capacity-v1,40,3,None,True
2,c900b3e8-e237-5531-895b-bfe6fa7851a4,7db3550a-82e7-573e-a7c7-1876dea74c09,0169f756-26d1-5ad6-acd4-4e6b0e319939,39c82c86-d06e-5f26-9712-a92b4018084e,5bb84918-2141-55b8-aab6-01257acfa73d,None,None,None,SUGERIDA,102.089955,...,None,None,None,103,None,POC-whole-cohort-ceil-capacity-v1,40,3,None,True
3,455ffcd7-bf42-5f21-9329-361cf19999fe,7db3550a-82e7-573e-a7c7-1876dea74c09,0169f756-26d1-5ad6-acd4-4e6b0e319939,519ead55-4294-52e1-b326-550e4be54cd0,53abed31-c5c0-504e-a7a9-f89f13a31fb3,None,None,None,SUGERIDA,103.245511,...,None,None,None,104,None,POC-whole-cohort-ceil-capacity-v1,40,3,None,True
4,74e6da20-22c3-55a3-88ae-097ce479d227,7db3550a-82e7-573e-a7c7-1876dea74c09,0169f756-26d1-5ad6-acd4-4e6b0e319939,519ead55-4294-52e1-b326-550e4be54cd0,8098c200-9822-56dc-a701-71f67f356b82,None,None,None,SUGERIDA,103.245511,...,None,None,None,104,None,POC-whole-cohort-ceil-capacity-v1,40,3,None,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,1294cbaa-7382-5b1c-b01b-4b8cc7e3b880,e4e8c58b-0aa4-5537-990c-38e4d8a240b9,fb031c8a-21cb-5aa1-bbcc-14b149dbcfb5,11e42499-2538-5286-a8b9-3fda55802247,d30d0d0a-6af3-5ba7-b60d-e25d687c1e69,None,None,None,SUGERIDA,96.586051,...,None,None,None,97,None,POC-whole-cohort-ceil-capacity-v1,40,3,None,True
86,0567967c-647d-56b0-b4e1-7a7dfa3087d2,e4e8c58b-0aa4-5537-990c-38e4d8a240b9,fb031c8a-21cb-5aa1-bbcc-14b149dbcfb5,11e42499-2538-5286-a8b9-3fda55802247,e7f05bc1-407f-5670-84d7-aae409217d6f,None,None,None,SUGERIDA,96.586051,...,None,None,None,97,None,POC-whole-cohort-ceil-capacity-v1,40,3,None,True
87,47ccb11e-0745-5fab-8ba8-f61e0594e7a5,e4e8c58b-0aa4-5537-990c-38e4d8a240b9,fb031c8a-21cb-5aa1-bbcc-14b149dbcfb5,1494b22f-3999-5714-ba12-3e8a1c65117c,5ebf59a4-4457-529e-98cb-04d804233a7c,None,None,None,SUGERIDA,96.497162,...,None,None,None,97,None,POC-whole-cohort-ceil-capacity-v1,40,3,None,True
88,b4676bbf-743d-5e8c-8c66-49be28ae642b,e4e8c58b-0aa4-5537-990c-38e4d8a240b9,fb031c8a-21cb-5aa1-bbcc-14b149dbcfb5,1494b22f-3999-5714-ba12-3e8a1c65117c,7064e2c5-ce43-55f0-b151-5d1dd109de0f,None,None,None,SUGERIDA,96.497162,...,None,None,None,97,None,POC-whole-cohort-ceil-capacity-v1,40,3,None,True


,rule_id,status,reason,evidence
0,POC-DRAFT,PASS,Proposta demonstrativa sem aprovação automática,proposal_generations.status
1,RF008-partial,PASS,Sem docente/horário/sala na representação inic...,offers.teacher_suggestion
2,RGN001/US027,NOT_EVALUATED,Decisão humana e prevalência de edições não ex...,DEMO scope
3,RGN002/RF010,NOT_EVALUATED,Integração institucional entre todos os cursos...,DEMO scope
4,RGN003/RF021–025/US029–030,NOT_EVALUATED,Alvo agregado; composição regular/dependência/...,DEMO scope
5,RGN023/US028,NOT_EVALUATED,Gate de validação da matriz depende da aplicação,DEMO scope
6,RF009/RF020/US026,NOT_EVALUATED,"Turno, coorte de ingresso e matriz homologada ...",DEMO scope
7,RF040–047/US033–034,NOT_EVALUATED,Capacidade DEMO não é regra institucional; uni...,DEMO scope
8,RF117–125,NOT_EVALUATED,Sem elegibilidade PRIME/Escola/turno ou aprova...,DEMO scope
9,RF155–159/US005–006,NOT_EVALUATED,SQL sintético e temporal; nenhuma integração i...,DEMO scope


4. Reconciliação somente leitura: linhas SQL, versões Registry, snapshot e bytes RustFS. Esperado: mesmos IDs, quantidades e hashes.

In [6]:
evidence["reconciliation"]=reconcile(cfg,proposal_id,pages,offers)
display(evidence["reconciliation"])
display(Markdown(f"[API](http://localhost:8000/v1/proposals/{proposal_id}/application-view) · [MLflow]({cfg.ui}) · [RustFS](http://localhost:9001)"))


{'sql': {'database': 'planejamento_db',
  'schema': 'planning',
  'status': 'PASS',
  'execution_id': '1ff2ee34-c872-47ae-a166-ae12d8f24ac5',
  'run_id': '6ac7e51b7b52456cb9b2230152691bed',
  'model_version': '17',
  'proposal_status': 'DRAFT',
  'stored_rows': 90,
  'proposed_students': 8694,
  'proposed_groups': 270,
  'query': 'sql/queries/verify_proposal.sql'},
 'storage': [{'bucket': 'pucpr-poc',
   'key': 'mlflow/4/models/m-29e446a82b6b42c482d0b0d89ddb8727/artifacts/MLmodel',
   'bytes': 1336,
   'sha256': '67170effc3266f094fa3f605b7416941736951200aeb5a3746b7cef05f7a3fe4',
   'object_version_id': '9da37cea-0e93-462a-a99e-d20b0fd4289d',
   'model_version': '17',
   'status': 'PASS'},
  {'bucket': 'pucpr-poc',
   'key': 'mlflow/4/models/m-29e446a82b6b42c482d0b0d89ddb8727/artifacts/model.pkl',
   'bytes': 8671,
   'sha256': '4268666504b258b661b9c6cc009bdf8c21d06fde4317fb389a585e633be3ad0b',
   'object_version_id': '35e102ba-90d6-45d1-867f-3d1e0fbb42c0',
   'model_version': '17',
   

[API](http://localhost:8000/v1/proposals/1ff2ee34-c872-47ae-a166-ae12d8f24ac5/application-view) · [MLflow](http://127.0.0.1:5000) · [RustFS](http://localhost:9001)

Guardar evidência desta execução e links. Esperado: caminho do relatório, sem credenciais.

In [7]:
report_status="PARTIAL" if any(isinstance(value,str) and value.startswith("NOT_EXECUTED") for value in evidence.values()) else "PASS"
if evidence.get("campaign",{}).get("status") in ("FAILED","PARTIAL","CANCELLED","PAUSED"):
    report_status="PARTIAL"
display(report(NOTEBOOK_ID,evidence,status=report_status,generation="v3",limitations=["Dados DEMO e contrato provisório; regras não avaliadas não são homologação", "Execução sem erro do notebook não implica execução de operações opt-in desativadas"]))


'/reports/v3/continuation-auto-check.json'